In [1]:
import cosmic.utils as cu
import pandas as pd
import numpy as np
from astropy.io import fits
from astropy.table import Table
import matplotlib.pyplot as plt

# 只用DESI DR1的BGS和LRG

In [38]:
path = './desi_dr1_BGS_LRG_xsdssdr19.fits'
df_raw = cu.readfile(path)

In [39]:
df_raw.columns

Index(['DESI_TARGETID', 'DESI_Z', 'DESI_ZERR', 'DESI_ZWARN', 'DESI_DELTACHI2',
       'DESI_RA', 'DESI_DEC', 'DESI_MASKBITS', 'DESI_DESI_TARGET',
       'DESI_BGS_TARGET', 'DESI_PRIORITY', 'SDSS_specobjid', 'SDSS_ra',
       'SDSS_dec', 'SDSS_z', 'SDSS_zErr', 'Separation'],
      dtype='object')

In [40]:
df = df_raw.copy()
df['ra'] = np.nan
df['dec'] = np.nan
df['z'] = np.nan
df['zErr'] = np.nan

# SDSS only
idx = (df['SDSS_ra'] > 0) & (df['DESI_RA'].isna())
df.loc[idx, 'ra'] = df.loc[idx, 'SDSS_ra']
df.loc[idx, 'dec'] = df.loc[idx, 'SDSS_dec']
df.loc[idx, 'z'] = df.loc[idx, 'SDSS_z']
df.loc[idx, 'zErr'] = df.loc[idx, 'SDSS_zErr']

# DESI only
idx = (df['SDSS_ra'].isna()) & (df['DESI_RA'] > 0)
df.loc[idx, 'ra'] = df.loc[idx, 'DESI_RA']
df.loc[idx, 'dec'] = df.loc[idx, 'DESI_DEC']
df.loc[idx, 'z'] = df.loc[idx, 'DESI_Z']
df.loc[idx, 'zErr'] = df.loc[idx, 'DESI_ZERR']

# Overlap
idx = (df['SDSS_ra'] > 0) & (df['DESI_RA'] > 0)
overlap = df[idx]
desi_better = overlap['DESI_ZERR'] <= overlap['SDSS_zErr']
sdss_better = overlap['DESI_ZERR'] > overlap['SDSS_zErr']

df.loc[overlap[desi_better].index, 'ra'] = overlap[desi_better]['DESI_RA']
df.loc[overlap[desi_better].index, 'dec'] = overlap[desi_better]['DESI_DEC']
df.loc[overlap[desi_better].index, 'z'] = overlap[desi_better]['DESI_Z']
df.loc[overlap[desi_better].index, 'zErr'] = overlap[desi_better]['DESI_ZERR']

df.loc[overlap[sdss_better].index, 'ra'] = overlap[sdss_better]['SDSS_ra']
df.loc[overlap[sdss_better].index, 'dec'] = overlap[sdss_better]['SDSS_dec']
df.loc[overlap[sdss_better].index, 'z'] = overlap[sdss_better]['SDSS_z']
df.loc[overlap[sdss_better].index, 'zErr'] = overlap[sdss_better]['SDSS_zErr']

cols = ['ra', 'dec', 'z', 'zErr', 
        'DESI_TARGETID', 'DESI_Z', 'DESI_ZERR', 'DESI_ZWARN', 'DESI_DELTACHI2',
        'DESI_RA', 'DESI_DEC', 'DESI_MASKBITS', 'DESI_DESI_TARGET',
        'DESI_BGS_TARGET', 'DESI_PRIORITY', 
        'SDSS_specobjid', 'SDSS_ra', 'SDSS_dec', 'SDSS_z', 'SDSS_zErr']
df = df[cols]
df = df.fillna(np.nan)
print(df.shape)

(10901607, 20)


In [41]:
# check Nan 
print(df[~(df['ra'] > 0)].shape)
print(df[~(df['z'] > 0)].shape)
print(df[~(df['zErr'] > 0)].shape)

(0, 20)
(4072, 20)
(958, 20)


In [42]:
# remove NaN
mask = (df['ra'] > 0)
mask &= (df['z'] > 0) 
mask &= (df['zErr'] > 0) & (df['zErr'] < 0.001)
df_save = df[mask]
print(df_save.shape)

(10887464, 20)


In [43]:
output_path = './DESIDR1BGSLRG_xSDSSDR19.fits'
cu.savefile(df_save, output_path)

# 重新清理的DESI DR1

In [2]:
path = './desidr1_xsdssdr19.fits'
df_raw = cu.readfile(path)

In [3]:
df_raw.columns

Index(['DESI_TARGETID', 'DESI_RA', 'DESI_DEC', 'DESI_Z', 'DESI_ZERR',
       'DESI_DELTACHI2', 'DESI_DESI_TARGET', 'DESI_BGS_TARGET',
       'DESI_PRIORITY', 'SDSS_specobjid', 'SDSS_ra', 'SDSS_dec', 'SDSS_z',
       'SDSS_zErr', 'Separation'],
      dtype='object')

In [8]:
df = df_raw.copy()
df['ra'] = np.nan
df['dec'] = np.nan
df['z'] = np.nan
df['zErr'] = np.nan

# SDSS only
idx = (df['SDSS_ra'] > 0) & (df['DESI_RA'].isna())
df.loc[idx, 'ra'] = df.loc[idx, 'SDSS_ra']
df.loc[idx, 'dec'] = df.loc[idx, 'SDSS_dec']
df.loc[idx, 'z'] = df.loc[idx, 'SDSS_z']
df.loc[idx, 'zErr'] = df.loc[idx, 'SDSS_zErr']

# DESI only
idx = (df['SDSS_ra'].isna()) & (df['DESI_RA'] > 0)
df.loc[idx, 'ra'] = df.loc[idx, 'DESI_RA']
df.loc[idx, 'dec'] = df.loc[idx, 'DESI_DEC']
df.loc[idx, 'z'] = df.loc[idx, 'DESI_Z']
df.loc[idx, 'zErr'] = df.loc[idx, 'DESI_ZERR']

# Overlap
idx = (df['SDSS_ra'] > 0) & (df['DESI_RA'] > 0)
overlap = df[idx]
desi_better = overlap['DESI_ZERR'] <= overlap['SDSS_zErr']
sdss_better = overlap['DESI_ZERR'] > overlap['SDSS_zErr']

df.loc[overlap[desi_better].index, 'ra'] = overlap[desi_better]['DESI_RA']
df.loc[overlap[desi_better].index, 'dec'] = overlap[desi_better]['DESI_DEC']
df.loc[overlap[desi_better].index, 'z'] = overlap[desi_better]['DESI_Z']
df.loc[overlap[desi_better].index, 'zErr'] = overlap[desi_better]['DESI_ZERR']

df.loc[overlap[sdss_better].index, 'ra'] = overlap[sdss_better]['SDSS_ra']
df.loc[overlap[sdss_better].index, 'dec'] = overlap[sdss_better]['SDSS_dec']
df.loc[overlap[sdss_better].index, 'z'] = overlap[sdss_better]['SDSS_z']
df.loc[overlap[sdss_better].index, 'zErr'] = overlap[sdss_better]['SDSS_zErr']

cols = ['ra', 'dec', 'z', 'zErr', 
        'DESI_TARGETID', 'DESI_RA', 'DESI_DEC', 'DESI_Z', 'DESI_ZERR',
        'DESI_DELTACHI2', 'DESI_DESI_TARGET', 'DESI_BGS_TARGET', 'DESI_PRIORITY', 
        'SDSS_specobjid', 'SDSS_ra', 'SDSS_dec', 'SDSS_z', 'SDSS_zErr',]
df = df[cols]
df = df.fillna(np.nan)
print(df.shape)

(11437514, 18)


In [9]:
# check Nan 
print(df[~(df['ra'] > 0)].shape)
print(df[~(df['z'] > 0)].shape)
print(df[~(df['zErr'] > 0)].shape)

(0, 18)
(1072, 18)
(958, 18)


In [10]:
# remove NaN
mask = (df['ra'] > 0)
mask &= (df['z'] > 0) 
mask &= (df['zErr'] > 0) & (df['zErr'] < 0.001)
df= df[mask]
print(f'After remove NaN: {len(df)}')

After remove NaN: 11426350


In [12]:
# 排除DESI与SDSS光谱红移差异大的星系
overlap_idx = (df['SDSS_z'] > 0) & (df['DESI_Z'] > 0)
overlap_data = df[overlap_idx]
z_diff_mask = abs(overlap_data['DESI_Z'] - overlap_data['SDSS_z']) > 0.005
exclude_idx = overlap_data[z_diff_mask].index
df_save = df.drop(exclude_idx)
print(f'After exclude DESI-SDSS z diff > 0.005: {len(df_save)}')

After exclude DESI-SDSS z diff > 0.005: 11423813


In [13]:
output_path = './DESIDR1_xSDSSDR19.fits'
cu.savefile(df_save, output_path)